# **Лабораторнгая работа №1 СИИиМО**



## Введение: 

Цель работы: изучение основ линейной регрессии, построение простейших моделей регрессии, обучение на реальных данных и оценка качества. 
Задачи:
1. загрузить датасет
2. подготовить данные для дальнейшей работы (провести первичный анализ, визуализировать распределение признаков и целевой переменной)
3. Провести предобработку данных
4. Построить матрицу корреляций. Сделать выводы о наличии мультиколлинеарности (расчет VIF-коэффициента)
5. Построить регрессионные модели (линейная, лассо и гребневая). Если целевая переменная - категориальная, то исследовать логистическую регрессию. Разделить на тренировочную и тестовую выборки (80/20 или 70/30). Использовать кросс-валидацию. Оценить качество построенной модели с помощью метрик: RMSE (Root Mean Square Error), R² (коэффициент детерминации) и MAPE (Mean Absolute Percentage Error).
6. Устранить мультиколлинеарность и снизить размерность признаков с помощью метода главных компонент (PCA). Перед проведением PCA провести стандартизацию данных.
7. Повторить шаг 5 (линейная, лассо и гребневая регрессия), но использовать в качестве признаков не исходные данные, а главные компоненты. Сравнить метрики качества (RMSE, R² и MAPE) моделей, обученных на исходных данных и на главных компонентах.

Столбцы:

date — дата наблюдения;
adj close — скорректированная цена закрытия;
close — цена закрытия (целевая переменная);
high, low, open — максимальная, минимальная и цена открытия;
volume — объём торгов;
ma_7, ma_30, ma_90 — скользящие средние;
daily_return — дневная доходность;
volatility_7, volatility_30 — волатильность;
rsi — индекс относительной силы;
macd, macd_signal — индикаторы MACD;
bb_upper, bb_lower — границы полос Боллинджера.

Этот набор данных содержит исторические данные о ежедневных ценах на золото, а также множество технических индикаторов, используемых для анализа и прогнозирования временных рядов. Он включает в себя основные рыночные показатели, такие как цена открытия, цена закрытия, максимальная и минимальная цены за день, скорректированная цена закрытия и объем торгов. Помимо необработанных данных о ценах, в наборе данных представлены расчетные показатели, такие как скользящие средние (за 7, 30 и 90 дней), дневная доходность, краткосрочная и долгосрочная волатильность, индекс относительной силы (RSI), MACD и его сигнальная линия, а также полосы Боллинджера (верхняя и нижняя). Благодаря этим расчетным показателям набор данных отлично подходит для моделей машинного обучения, особенно для задач прогнозирования цен и анализа трендов.

Все признаки, кроме date, — некатегориальные (числовые);
adj close и ma_* являются производными от close → утечка данных, их нужно исключить;
Признаки сильно коррелируют - ожидается мультиколлинеарность.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Библиотеки успешно загружены")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Библиотеки успешно загружены
pandas: 2.3.3
numpy: 2.2.6



Загружены все необходимые библиотеки:
- **pandas, numpy** — для работы с данными
- **matplotlib, seaborn** — для визуализации
- **sklearn** — модели, метрики, предобработка, кросс-валидация, PCA
- **statsmodels** — расчёт VIF-коэффициентов

Версии библиотек зафиксированы в выводе ячейки выше. Все библиотеки успешно загружены, ошибок нет. Отключены предупреждения для чистоты вывода.

In [2]:
df = pd.read_csv('gold_price_dts.csv', parse_dates=['date'])
print(f"Размер датасета: {df.shape}")
print(f"Количество наблюдений: {df.shape[0]}")
print(f"Количество признаков: {df.shape[1]} \n")
print(f"Период данных: {df['date'].min().date()} — {df['date'].max().date()}")
print("Информация о датасете:")
print(df.info())

print(f"\n\nОбщее количество пропущенных значений: {df.isnull().sum().sum()}")
duplicates = df.duplicated().sum()
print(f"Количество дубликатов: {duplicates}")

# Описательные статистики
print("\nОписательные статистики:")
df.describe().T

Размер датасета: (1167, 18)
Количество наблюдений: 1167
Количество признаков: 18 

Период данных: 2021-06-11 — 2026-01-30
Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1167 entries, 0 to 1166
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           1167 non-null   datetime64[ns]
 1   adj close      1167 non-null   float64       
 2   close          1167 non-null   float64       
 3   high           1167 non-null   float64       
 4   low            1167 non-null   float64       
 5   open           1167 non-null   float64       
 6   volume         1167 non-null   int64         
 7   ma_7           1167 non-null   float64       
 8   ma_30          1167 non-null   float64       
 9   ma_90          1167 non-null   float64       
 10  daily_return   1167 non-null   float64       
 11  volatility_7   1167 non-null   float64       
 12  volatility_30  1167 non-null 

,count,mean,min,25%,50%,75%,max,std
date,1167,2023-10-05 16:37:01.079691520,2021-06-11 00:00:00,2022-08-08 12:00:00,2023-10-05 00:00:00,2024-12-02 12:00:00,2026-01-30 00:00:00,NaN
adj close,1167.0,2363.105823,1623.300049,1827.299988,1985.900024,2666.099976,5318.399902,749.127289
close,1167.0,2363.105823,1623.300049,1827.299988,1985.900024,2666.099976,5318.399902,749.127289
high,1167.0,2375.595715,1623.300049,1831.600037,1996.199951,2675.449951,5586.200195,757.244374
low,1167.0,2349.989545,1615.099976,1816.800049,1979.800049,2654.599976,5301.600098,741.042344
open,1167.0,2362.322109,1620.400024,1825.25,1989.0,2662.400024,5415.700195,749.389242
volume,1167.0,4170.995716,0.0,88.0,297.0,867.5,209783.0,21394.527815
ma_7,1167.0,2354.938862,1643.671422,1827.585711,1986.785697,2654.771449,5070.914272,735.998718
ma_30,1167.0,2327.283295,1659.240002,1830.560004,1978.813334,2659.524996,4620.783317,697.354115
ma_90,1167.0,2263.573357,1705.499999,1813.691667,1948.046665,2588.156108,4252.53221,620.00894


Размер датасета: (1167, 18)
Количество наблюдений: 1167
Количество признаков: 18 

Период данных: 2021-06-11 — 2026-01-30
Информация о датасете:
RangeIndex: 1167 entries, 0 to 1166
![alt text](image-1.png)


Общее количество пропущенных значений: 0
Количество дубликатов: 0

Описательные статистики:
![alt text](image.png)

mean - среднее
std - стандартное отклонение

Датасет успешно загружен с помощью `pd.read_csv()` с параметром `parse_dates=['date']` — это автоматически преобразует столбец с датами в формат `datetime64`, что необходимо для корректной работы с временными рядами.

**Ключевые характеристики датасета** (см. вывод выше):
- Общее количество наблюдений и признаков зафиксировано в выводе
- Период данных охватывает несколько лет — от начальной до конечной даты
- Структура данных: 17 столбцов, включая дату, цены, объёмы и технические индикаторы

Целевая переменная — `close` (цена закрытия золота). Первые строки показывают корректность загрузки — даты распознаны, числовые значения в правильном формате.

### Пояснение к проверке данных

Все столбцы имеют числовой тип (float64) или datetime. 
**Пропущенные значения отсутствуют** — это хорошая новость, не требуется дополнительная обработка пропусков.

Проверим наличие дубликатов и описательные статистики.

### Пояснение к описательным статистикам

Из описательных статистик (см. таблицу выше) видно:

**Диапазон цен:**
- Целевая переменная `close` варьируется в широком диапазоне — от минимального до максимального значения, указанных в таблице
- Средняя цена закрытия и медиана близки, что говорит о симметричности распределения
- Стандартное отклонение показывает высокую волатильность цен за период

**Различия масштабов признаков:**
- `volume` измеряется в тысячах/десятках тысяч (объёмы торгов)
- `daily_return` — в долях единицы (порядка 0.001–0.05)
- `rsi` — в диапазоне 0–100
- Цены — в сотнях/тысячах долларов

**Критически важно:** признаки имеют **разный масштаб**, поэтому перед применением PCA необходима **стандартизация** (StandardScaler). Без неё признаки с большим разбросом (например, цены) будут доминировать в главных компонентах, искажая результат.